# H65 Stage 0 — Colab CUDA

Chọn **Kernel → Colab → Auto Connect** trước khi chạy. Notebook này chỉ benchmark Stage 0 và sẽ dừng nếu CUDA/gate không đạt; chưa tạo output hoặc ZIP. Vì repository là private, cell 2 sẽ yêu cầu một GitHub fine-grained token có quyền **Contents: Read-only** cho repository `minggu151623/ViettelAIRACE`. Token được nhập ẩn và không được lưu vào Git remote.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/ViettelAIRACE')
assert (DRIVE_ROOT / 'turn2/input').exists(), 'Đặt turn2/input vào MyDrive/ViettelAIRACE trước'
print('Drive root:', DRIVE_ROOT)

In [ ]:
import base64
import getpass
import os
import shutil
import subprocess

repo_dir = Path('/content/ViettelAIRACE')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

# Repo là private. Token được nhập ẩn, chỉ truyền qua HTTP header cho lần
# clone này và không được lưu trong notebook hoặc Git remote.
github_token = getpass.getpass('GitHub fine-grained token (Contents: Read-only): ')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
clone_env = os.environ.copy()
clone_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.extraHeader',
    'GIT_CONFIG_VALUE_0': f'Authorization: Basic {basic_auth}',
})
clone_result = subprocess.run(
    [
        'git', 'clone', '--branch', 'codex/core-rebuild-h57', '--single-branch',
        'https://github.com/minggu151623/ViettelAIRACE.git', str(repo_dir),
    ],
    text=True, capture_output=True, env=clone_env,
)
github_token = None
basic_auth = None
clone_env['GIT_CONFIG_VALUE_0'] = ''
if clone_result.returncode != 0:
    print(clone_result.stderr)
    raise RuntimeError('Không clone được repo private; kiểm tra token và quyền Contents: Read-only.')

os.chdir(repo_dir)
(repo_dir / 'turn2').mkdir(parents=True, exist_ok=True)
(repo_dir / 'experiments/H65_crosslingual_projection_corrected_execution/results').mkdir(parents=True, exist_ok=True)
(repo_dir / 'external').mkdir(parents=True, exist_ok=True)
target_input = repo_dir / 'turn2/input'
if target_input.exists():
    shutil.rmtree(target_input)
shutil.copytree(DRIVE_ROOT / 'turn2/input', target_input)
input_files = sorted(target_input.glob('*.txt'))
assert len(input_files) == 100, f'Expected 100 input files, found {len(input_files)}'
subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/VinAIResearch/PhoNER_COVID19.git', 'external/PhoNER_COVID19'],
    check=True,
)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository ready:', repo_dir, commit)

In [ ]:
!pip -q install -U transformers sentencepiece accelerate huggingface_hub
import torch
assert torch.cuda.is_available(), 'CUDA chưa bật: Runtime → Change runtime type → GPU'
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
!python experiments/H65_crosslingual_projection_corrected_execution/run_stage_0_colab.py \
  --phoner-dev external/PhoNER_COVID19/data/word/dev_word.json \
  --input turn2/input \
  --output experiments/H65_crosslingual_projection_corrected_execution/results

In [ ]:
from pathlib import Path
import json
report = json.loads(Path('experiments/H65_crosslingual_projection_corrected_execution/results/stage_0_report.json').read_text())
display(report)
print('Copy this report back to Codex. Do not run Turn-2 if status is FAIL.')